# Add text type tasks to LabelStudio

NOTE: Right now this loads to a "test" project so as not to overwrite existing text_type annotations; eventually this should be updated so it loads to the real tasks.

In [1]:
import pandas as pd
import numpy as np
import os
import json
from dotenv import load_dotenv
from label_studio_sdk.client import LabelStudio
import sys

# Load environment variables
load_dotenv()

# Direct the notebook to find scripts in parent directory, since it is sitting one level down
os.chdir("..")

sys.path.append(os.getcwd())

# Import the new task classes
from adt_labelstudio.text_type import TextTypeTask
from adt_labelstudio.utils import get_project_annotations, get_ls_project_id_from_name

# Utility functions for normalization (if not already available from utils)
from adt_eval.utils.transcript_cleaner import standardize_transcript, normalize_transcript


In [2]:

# Connect to the Label Studio API and check the connection
LABEL_STUDIO_URL =  "https://" + os.getenv("LABEL_STUDIO_HOST")
API_KEY = os.getenv("LABEL_STUDIO_TOKEN")

ls_client = LabelStudio(base_url=LABEL_STUDIO_URL, api_key=API_KEY)

text_type_task = TextTypeTask()

In [3]:
# Specify where the LLM logs are located
file_dir = "output/eval_feb_1/logs/text_extraction/"
file_list = os.listdir(file_dir)

source_project_name = "A1: Text Extraction"
target_project_name = "TEST: Text type" # Adding to test project so as not to overwrite existing annotations


# Get annotations from previous task, used to populate this one
gs_annotations = get_project_annotations(ls_client, project_name=source_project_name)

# Get annotations that have already been done for this task
target_project_tasks = get_project_annotations(ls_client, project_name=target_project_name)

tasks_to_load = []

for f in file_list:

    llm_log, book_name, page_id = text_type_task.get_llm_log(f"{file_dir}/{f}")

    # Confirm that task does not already exist
    if target_project_tasks.shape[0]>0 and (book_name, page_id) in [xy for xy in zip(target_project_tasks['book_id'], target_project_tasks['page_id'])]:
        print(f"Task for {book_name} page {page_id} already exists in LabelStudio and was not added.")
        continue

    print(f"Processing task for {book_name} page {page_id}.")

    # Read in the LLM log file to dataframe
    llm_df = text_type_task.load_llm_log_to_df(llm_log)

    # Get the Gold Standard data as dataframe
    gs_annotation = text_type_task.get_single_annotation(gs_annotations, book_name, page_id)
    gs_df = text_type_task.load_gs_annotation_to_df(gs_annotation)

    # Merge Gold Standard and LLM dataframes onexact match
    matched_df = text_type_task.merge_gs_with_llm(gs_df, llm_df)

    # Create labelstudio task, consisting of input data and predictions
    task_data = text_type_task.populate_task_data(gs_annotation)
    task_predictions = text_type_task.populate_task_predictions(matched_df)
    task_json = text_type_task.create_one_task(task_data, task_predictions)

    tasks_to_load.append(task_json)

# Add the whole list to LabelStudio
if len(tasks_to_load) > 0:
    target_project_id  = get_ls_project_id_from_name(ls_client, project_name=target_project_name)
    ls_client.projects.import_tasks(
                id=target_project_id,
                request=tasks_to_load,
            )
with open("adt_labelstudio/tasks_to_upload/text_type_tasks.json", "w") as f:                  
    json.dump(tasks_to_load, f)

Processing task for B-54 page 1.
Processing task for B-54 page 3.
Processing task for B-89 page 3.
Processing task for B-89 page 7.
Processing task for B-89 page 9.
Processing task for B-89 page 14.
Processing task for B-89 page 16.
Processing task for B-89 page 17.
Processing task for B-89 page 23.
Processing task for B-89 page 25.
Processing task for B-89 page 30.
Processing task for B-89 page 32.
Processing task for B-89 page 51.
Processing task for B-89 page 55.
Processing task for B-89 page 70.
Processing task for C-1 page 9.
Processing task for C-30 page 0.
Processing task for C-30 page 2.
Processing task for C-30 page 4.
Processing task for C-30 page 7.
Processing task for C-30 page 9.
Processing task for C-30 page 12.
Processing task for C-30 page 22.
Processing task for C-30 page 28.
Processing task for C-30 page 29.
Processing task for C-45 page 0.
Processing task for C-45 page 5.
Processing task for C-45 page 6.
Processing task for C-49 page 0.
Processing task for C-49 page 

In [4]:
task_json

{'data': {'book_id': 'W-38',
  'page_id': 0,
  'page_image': 'azure-blob://adt-pipeline/evaluation/gold_standard/pages/W-38__page_1.png',
  'page_text_all': 'U.S. EDITION\n\nPRIMARY MATHEMATICS 1A\n\nTEXTBOOK\n\nMarshall Cavendish Education\n\nSingaporeMath.com Inc'},
 'predictions': [{'result': [{'value': {'text': 'us edition',
      'taxonomy': [['book_metadata']],
      'start': 0,
      'end': 12},
     'from_name': 'text_type_annotations',
     'to_name': 'page_text_all',
     'type': 'taxonomy'},
    {'value': {'text': 'primary mathematics 1a',
      'taxonomy': [['']],
      'start': 14,
      'end': 36},
     'from_name': 'text_type_annotations',
     'to_name': 'page_text_all',
     'type': 'taxonomy'},
    {'value': {'text': 'textbook',
      'taxonomy': [['book_subtitle']],
      'start': 38,
      'end': 46},
     'from_name': 'text_type_annotations',
     'to_name': 'page_text_all',
     'type': 'taxonomy'},
    {'value': {'text': 'marshall cavendish education',
      'tax